# Mini-Courier Planner

## Capstone implementation walkthrough

This notebook demonstrates the complete integrated courier-planning system required by the capstone assignment. It covers parcel prioritization, 0/1 Knapsack dynamic programming, Dijkstra shortest paths, nearest-neighbor TSP approximation, and runtime benchmarking.

The implementation is split into modules under `src/` so each algorithm can be tested independently.

In [1]:
from pathlib import Path
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / "datasets").exists() and (ROOT.parent / "datasets").exists():
    ROOT = ROOT.parent

from src.data_loader import load_parcels, load_graph, load_coordinates
from src.planner import CourierPlanner
from src.reporting import render_plan
from src.visualize import plot_route, plot_benchmark
from src.benchmark import run_benchmark


## 1. Load the datasets

The parcel file stores parcel properties, the route file stores weighted graph edges, and the locations file provides coordinates for the route diagram.

In [2]:
parcels = load_parcels(ROOT / "datasets" / "parcels.csv")
graph = load_graph(ROOT / "datasets" / "routes.csv")
coordinates = load_coordinates(ROOT / "datasets" / "locations.csv")

print(f"Parcels loaded: {len(parcels)}")
print(f"Graph nodes: {len(list(graph.nodes()))}")
print("Vehicle capacity: 15 units")


Parcels loaded: 10
Graph nodes: 11
Vehicle capacity: 15 units


## 2. Parcel prioritization with merge sort

Sort key: higher priority first, earlier deadline first, then higher value as the final tie-breaker. Complexity: `O(n log n)`.

In [3]:
from src.sorting import merge_sort_parcels

sorted_parcels = merge_sort_parcels(parcels)
for p in sorted_parcels:
    print(f"{p.parcel_id}: destination={p.destination}, weight={p.weight}, priority={p.priority}, deadline={p.deadline}, value={p.value}")


P03: destination=C3, weight=3, priority=10, deadline=3, value=50
P01: destination=C1, weight=4, priority=9, deadline=2, value=70
P06: destination=C6, weight=7, priority=9, deadline=5, value=95
P02: destination=C2, weight=6, priority=8, deadline=1, value=90
P08: destination=C8, weight=3, priority=8, deadline=3, value=60
P04: destination=C4, weight=5, priority=7, deadline=4, value=80
P10: destination=C10, weight=2, priority=7, deadline=5, value=40
P05: destination=C5, weight=2, priority=6, deadline=2, value=35
P07: destination=C7, weight=4, priority=5, deadline=6, value=55
P09: destination=C9, weight=5, priority=4, deadline=7, value=65


## 3. Vehicle loading with 0/1 Knapsack

Each parcel is either selected once or not selected. The DP table stores the best achievable value for each prefix of parcels and each capacity from 0 to 15.

In [4]:
from src.knapsack import zero_one_knapsack

packing = zero_one_knapsack(sorted_parcels, capacity=15)
print("Selected parcels:", [p.parcel_id for p in packing.selected_parcels])
print("Total weight:", packing.total_weight)
print("Total value:", packing.total_value)


Selected parcels: ['P03', 'P08', 'P04', 'P10', 'P05']
Total weight: 15
Total value: 265


In [5]:
# DP table
print("DP table (rows = parcels considered, columns = capacity)")
header = "     " + " ".join(f"{w:3}" for w in range(16))
print(header)
for i, row in enumerate(packing.dp_table):
    label = "0" if i == 0 else sorted_parcels[i-1].parcel_id
    print(f"{label:>4} " + " ".join(f"{value:3}" for value in row))


DP table (rows = parcels considered, columns = capacity)
       0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
 P03   0   0   0  50  50  50  50  50  50  50  50  50  50  50  50  50
 P01   0   0   0  50  70  70  70 120 120 120 120 120 120 120 120 120
 P06   0   0   0  50  70  70  70 120 120 120 145 165 165 165 215 215
 P02   0   0   0  50  70  70  90 120 120 140 160 165 165 210 215 215
 P08   0   0   0  60  70  70 110 130 130 150 180 180 200 220 225 225
 P04   0   0   0  60  70  80 110 130 140 150 180 190 210 220 230 260
 P10   0   0  40  60  70 100 110 130 150 170 180 190 220 230 250 260
 P05   0   0  40  60  75 100 110 135 150 170 185 205 220 230 255 265
 P07   0   0  40  60  75 100 110 135 150 170 185 205 220 230 255 265
 P09   0   0  40  60  75 100 110 135 150 170 185 205 220 230 255 265


## 4. Shortest path with Dijkstra

The graph has non-negative edge weights, so Dijkstra is suitable. The heap implementation uses adjacency lists.

In [6]:
shortest = graph.dijkstra("D", "C6")
print("Path:", " -> ".join(shortest.path))
print(f"Distance: {shortest.distance:.2f} km")


Path: D -> C3 -> C6
Distance: 10.00 km


## 5. Overall route with nearest-neighbor TSP approximation

The planner repeatedly chooses the unvisited destination with the smallest Dijkstra distance from the current location, then returns to the depot. This is a heuristic, so optimality is not guaranteed.

In [7]:
planner = CourierPlanner(graph=graph, capacity=15)
result = planner.plan(parcels=parcels, depot="D")
print(render_plan(result))


MINI-COURIER PLANNER
Parcel prioritization
----------------------------------------------------------------------
ID     Destination  Weight  Priority  Deadline  Value
P03    C3                3       10         3     50
P01    C1                4        9         2     70
P06    C6                7        9         5     95
P02    C2                6        8         1     90
P08    C8                3        8         3     60
P04    C4                5        7         4     80
P10    C10               2        7         5     40
P05    C5                2        6         2     35
P07    C7                4        5         6     55
P09    C9                5        4         7     65

Packing plan (0/1 Knapsack)
----------------------------------------------------------------------
Selected: P03, P08, P04, P10, P05
Total weight : 15 / 15
Total value  : 265

Route plan (Dijkstra + Nearest-Neighbor TSP)
----------------------------------------------------------------------
Delivery 

In [8]:
route_image = ROOT / "images" / "route_visualization.png"
plot_route(graph, coordinates, result.route, route_image)
print(route_image)


/mnt/data/mini_courier_planner/images/route_visualization.png


## 6. Runtime benchmarking

Benchmarks use a fixed random seed so the generated test inputs are reproducible. Runtime is measured with `time.perf_counter()` and averaged over repeated runs.

In [9]:
benchmark_results = run_benchmark()
for algorithm, points in benchmark_results.items():
    print(f"\n{algorithm}")
    for size, seconds in points:
        print(f"  size={size:>3}  average={seconds*1000:.4f} ms")

benchmark_image = ROOT / "images" / "runtime_comparison.png"
plot_benchmark(benchmark_results, benchmark_image)
print("\nRuntime chart:", benchmark_image)



sorting
  size=100  average=0.2261 ms
  size=200  average=0.8442 ms
  size=400  average=1.2924 ms
  size=800  average=2.2863 ms
  size=1600  average=5.7439 ms

knapsack
  size=100  average=1.4101 ms
  size=200  average=4.7469 ms
  size=400  average=14.4826 ms
  size=800  average=61.2630 ms
  size=1600  average=240.2068 ms

route_planning
  size=  5  average=0.0465 ms
  size=  8  average=0.1283 ms
  size= 11  average=0.3822 ms
  size= 14  average=0.9047 ms
  size= 17  average=2.1017 ms



Runtime chart: /mnt/data/mini_courier_planner/images/runtime_comparison.png


## 7. Integrated result summary

The complete program can also be run from the command line with `python -m src.main`. Unit tests are provided under `tests/`.